# 상호작용 (Interaction)

_“그래픽은 단번에 '그려지는' 것이 아닙니다. 데이터의 상호작용에 의해 형성된 모든 관계가 드러날 때까지 '구성'되고 재구성됩니다. 가장 좋은 그래픽 작업은 의사 결정자 자신이 수행하는 작업입니다.”_ &mdash; [Jacques Bertin](https://books.google.com/books?id=csqX_xnm4tcC)

시각화는 데이터를 이해하는 강력한 수단을 제공합니다. 그러나 단일 이미지는 일반적으로 기껏해야 몇 가지 질문에 대한 답을 제공할 뿐입니다. *상호작용(interaction)*을 통해 우리는 정적인 이미지를 탐색을 위한 도구로 변환할 수 있습니다. 관심 지점을 강조 표시하고, 더 미세한 패턴을 드러내기 위해 확대하고, 여러 뷰를 연결하여 다차원적 관계에 대해 추론할 수 있습니다.

상호작용의 핵심은 *선택(selection)*이라는 개념입니다. 이는 우리가 관심 있는 요소나 영역을 컴퓨터에 알리는 수단입니다. 예를 들어, 점 위에 마우스를 올리거나, 여러 마크를 클릭하거나, 영역 주위에 경계 상자를 그려 추가 조사를 위해 데이터의 하위 세트를 강조 표시할 수 있습니다.

시각적 인코딩 및 데이터 변환과 함께 Altair는 상호작용을 작성하기 위한 *선택* 추상화를 제공합니다. 이러한 선택은 세 가지 측면을 포함합니다:

1. 마우스 호버, 클릭, 드래그, 스크롤 및 터치 이벤트와 같이 관심 있는 지점이나 영역을 선택하기 위한 입력 이벤트 처리.
2. 입력을 일반화하여 주어진 데이터 레코드가 선택 범위 내에 있는지 여부를 결정하는 선택 규칙(또는 [*서술어(predicate)*](https://en.wikipedia.org/wiki/Predicate_%28mathematical_logic%29))을 형성.
3. 선택 서술어를 사용하여 *조건부 인코딩*, *필터 변환* 또는 *스케일 도메인*을 구동함으로써 시각화를 동적으로 구성.

이 노트북은 대화형 선택을 소개하고 이를 사용하여 동적 쿼리, 이동 및 확대/축소, 세부 정보 표시, 브러싱 및 연결과 같은 다양한 상호작용 기술을 작성하는 방법을 탐구합니다.

_이 노트북은 [데이터 시각화 커리큘럼](https://github.com/uwdata/visualization-curriculum)의 일부입니다._

In [1]:
import pandas as pd
import altair as alt

## 데이터 세트 (Datasets)

우리는 [vega-datasets](https://github.com/vega/vega-datasets) 컬렉션의 다양한 데이터 세트를 시각화할 것입니다:

- 1970년대와 1980년대 초반의 자동차(`cars`) 데이터 세트,
- 이전 [데이터 변환](https://github.com/uwdata/visualization-curriculum/blob/master/altair_data_transformation.ipynb) 노트북에서 사용된 영화(`movies`) 데이터 세트,
- 10년 동안의 [S&P 500](https://en.wikipedia.org/wiki/S%26P_500_Index) (`sp500`) 주가 데이터 세트,
- 기술 기업의 주식(`stocks`) 데이터 세트,
- 출발 시간, 거리 및 도착 지연을 포함한 비행(`flights`) 데이터 세트.

In [2]:
cars = 'https://cdn.jsdelivr.net/npm/vega-datasets@1/data/cars.json'
movies = 'https://cdn.jsdelivr.net/npm/vega-datasets@1/data/movies.json'
sp500 = 'https://cdn.jsdelivr.net/npm/vega-datasets@1/data/sp500.csv'
stocks = 'https://cdn.jsdelivr.net/npm/vega-datasets@1/data/stocks.csv'
flights = 'https://cdn.jsdelivr.net/npm/vega-datasets@1/data/flights-5k.json'

## 선택 소개 (Introducing Selections)

기본적인 선택부터 시작하겠습니다: 점을 클릭하여 강조 표시하는 것입니다. 자동차(`cars`) 데이터 세트를 사용하여 마력(Horsepower) 대 갤런당 마일(Miles_per_Gallon)의 산점도로 시작하고, 자동차 엔진의 실린더 수에 대한 색상 인코딩을 적용합니다.

또한 `alt.selection_single()`을 호출하여 *단일 값*에 대해 정의된 선택을 원하는 선택 인스턴스를 생성합니다. 기본적으로 선택은 마우스 클릭을 사용하여 선택된 값을 결정합니다. 차트에 선택을 등록하려면 `.add_selection()` 메서드를 사용하여 추가해야 합니다.

선택이 정의되면 이를 *조건부 인코딩*의 매개변수로 사용할 수 있습니다. 조건부 인코딩은 데이터 레코드가 선택 범위 내에 있는지 여부에 따라 다른 인코딩을 적용합니다. 예를 들어 다음 코드를 살펴보세요:

~~~ python
color=alt.condition(selection, 'Cylinders:O', alt.value('grey'))
~~~

이 인코딩 정의는 `selection` 내에 포함된 데이터 포인트는 `Cylinder` 필드에 따라 색상이 지정되어야 하고, 선택되지 않은 데이터 포인트는 기본값인 `grey`(회색)를 사용해야 함을 나타냅니다. 비어 있는 선택(선택된 것이 없는 상태)은 *모든* 데이터 포인트를 포함하므로 처음에는 모든 포인트에 색상이 지정됩니다.

*아래 차트에서 다른 점들을 클릭해 보세요. 어떤 일이 일어나나요? (배경을 클릭하면 선택 상태가 지워지고 "비어 있는" 선택 상태로 돌아갑니다.)*

In [3]:
selection = alt.selection_single();
  
alt.Chart(cars).mark_circle().add_selection(
    selection
).encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color=alt.condition(selection, 'Cylinders:O', alt.value('grey')),
    opacity=alt.condition(selection, alt.value(0.8), alt.value(0.1))
)

alt.Chart(...)

물론 데이터 포인트를 하나씩 강조 표시하는 것은 특별히 흥미롭지 않습니다! 하지만 앞으로 살펴보겠지만, 단일 값 선택은 더 강력한 상호작용을 위한 유용한 구성 요소가 됩니다. 또한 단일 값 선택은 Altair에서 제공하는 세 가지 선택 유형 중 하나일 뿐입니다:

- `selection_single` - 단일 이산 값을 선택합니다. 기본적으로 클릭 이벤트로 작동합니다.
- `selection_multi` - 여러 이산 값을 선택합니다. 마우스 클릭으로 첫 번째 값을 선택하고, Shift-클릭을 사용하여 추가 값을 선택하거나 해제합니다.
- `selection_interval` - 마우스 드래그로 시작되는 연속적인 값 범위를 선택합니다.

이러한 각 선택 유형을 나란히 비교해 보겠습니다. 코드를 깔끔하게 유지하기 위해 먼저 위와 같은 산점도 사양을 생성하는 함수(`plot`)를 정의하겠습니다. 선택 인스턴스를 `plot` 함수에 전달하여 차트에 적용할 수 있습니다:

In [4]:
def plot(selection):
    return alt.Chart(cars).mark_circle().add_selection(
        selection
    ).encode(
        x='Horsepower:Q',
        y='Miles_per_Gallon:Q',
        color=alt.condition(selection, 'Cylinders:O', alt.value('grey')),
        opacity=alt.condition(selection, alt.value(0.8), alt.value(0.1))
    ).properties(
        width=240,
        height=180
    )

`plot` 함수를 사용하여 선택 유형당 하나씩, 세 가지 차트 변형을 만들어 보겠습니다.

첫 번째(`single`) 차트는 이전 예제를 복제한 것입니다. 두 번째(`multi`) 차트는 Shift-클릭 상호작용을 통해 선택 항목 내에 여러 점을 포함하거나 제외하는 기능을 지원합니다. 세 번째(`interval`) 차트는 마우스 드래그 시 선택 영역(또는 *브러시(brush)*)을 생성합니다. 생성된 브러시를 드래그하여 다른 점들을 선택하거나, 커서가 브러시 내부에 있을 때 스크롤하여 브러시 크기를 조절(확대/축소)할 수 있습니다.

*아래의 각 차트와 상호작용해 보세요!*

In [5]:
alt.hconcat(
  plot(alt.selection_single()).properties(title='Single (Click)'),
  plot(alt.selection_multi()).properties(title='Multi (Shift-Click)'),
  plot(alt.selection_interval()).properties(title='Interval (Drag)')
)

alt.HConcatChart(...)

위의 예제는 각 선택 유형에 대해 기본 상호작용(클릭, Shift-클릭, 드래그)을 사용합니다. [Vega 이벤트 선택자 구문](https://vega.github.io/vega/docs/event-streams/)을 사용하여 입력 이벤트 사양을 제공함으로써 상호작용을 추가로 사용자 정의할 수 있습니다. 예를 들어, `single` 및 `multi` 차트가 `click` 이벤트 대신 `mouseover` 이벤트에서 트리거되도록 수정할 수 있습니다.

*두 번째 차트에서 Shift 키를 누른 채로 마우스를 움직여 데이터로 "그림을 그려" 보세요!*

In [6]:
alt.hconcat(
  plot(alt.selection_single(on='mouseover')).properties(title='단일 선택 (마우스 오버)'),
  plot(alt.selection_multi(on='mouseover')).properties(title='다중 선택 (Shift-마우스 오버)')
)

alt.HConcatChart(...)

Altair 선택의 기본 사항을 다루었으므로, 이제 이를 통해 가능한 다양한 상호작용 기술을 살펴보겠습니다!

## 동적 쿼리 (Dynamic Queries)

*동적 쿼리*는 관심 있는 패턴을 격리하기 위해 데이터를 빠르고 가역적으로 탐색할 수 있게 합니다. [Ahlberg, Williamson, & Shneiderman](https://www.cs.umd.edu/~ben/papers/Ahlberg1992Dynamic.pdf)이 정의한 바와 같이, 동적 쿼리는 다음을 특징으로 합니다:

- 쿼리를 그래픽으로 표현합니다.
- 쿼리 범위에 대한 가시적인 제한을 제공합니다.
- 데이터 및 쿼리 결과의 그래픽 표현을 제공합니다.
- 모든 쿼리 조정 후 결과에 대한 즉각적인 피드백을 제공합니다.
- 초보 사용자도 적은 교육으로 작업을 시작할 수 있도록 합니다.

일반적인 접근 방식은 슬라이더, 라디오 버튼, 드롭다운 메뉴와 같은 표준 사용자 인터페이스 위젯을 사용하여 쿼리 매개변수를 조작하는 것입니다. 동적 쿼리 위젯을 생성하기 위해 선택의 `bind` 작업을 쿼리하려는 하나 이상의 데이터 필드에 적용할 수 있습니다.

동적 쿼리를 사용하여 디스플레이를 필터링하는 대화형 산점도를 만들어 보겠습니다. 영화 평점(Rotten Tomatoes 및 IMDB)의 산점도가 주어지면, `Major_Genre` 필드에 선택을 추가하여 영화 장르별로 대화형 필터링을 활성화할 수 있습니다.

시작하기 위해, `movies` 데이터에서 고유한(null이 아닌) 장르를 추출해 보겠습니다:

In [7]:
df = pd.read_json(movies) # 영화 데이터 로드
genres = df['Major_Genre'].unique() # 고유한 필드 값 가져오기
genres = list(filter(lambda d: d is not None, genres)) # None 값 필터링
genres.sort() # 알파벳순으로 정렬

나중에 사용하기 위해 고유한 `MPAA_Rating` 값 목록도 정의해 보겠습니다:

In [8]:
mpaa = ['G', 'PG', 'PG-13', 'R', 'NC-17', 'Not Rated']

이제 드롭다운 메뉴에 바인딩된 `single` 선택을 만들어 보겠습니다.

*아래의 동적 쿼리 메뉴를 사용하여 데이터를 탐색해 보세요. 장르별로 평점이 어떻게 다른가요? `Major_Genre` 대신 `MPAA_Rating`(G, PG, PG-13 등)을 필터링하도록 코드를 어떻게 수정하시겠습니까?*

In [9]:
selectGenre = alt.selection_single(
    name='Select', # 선택 항목의 이름을 'Select'로 지정
    fields=['Major_Genre'], # 선택 항목을 Major_Genre 필드로 제한
    init={'Major_Genre': genres[0]}, # 첫 번째 장르 항목을 초기값으로 사용
    bind=alt.binding_select(options=genres) # 고유한 장르 값 메뉴에 바인딩
)

alt.Chart(movies).mark_circle().add_selection(
    selectGenre
).encode(
    x='Rotten_Tomatoes_Rating:Q',
    y='IMDB_Rating:Q',
    tooltip='Title:N',
    opacity=alt.condition(selectGenre, alt.value(0.75), alt.value(0.05))
)

alt.Chart(...)

위의 구성은 선택의 여러 측면을 활용합니다:

- 선택에 이름(`'Select'`)을 부여합니다. 이 이름은 필수는 아니지만, 생성된 동적 쿼리 메뉴의 레이블 텍스트에 영향을 줄 수 있습니다. (*이름을 제거하면 어떻게 될까요? 시도해 보세요!*)
- 선택을 특정 데이터 필드(`Major_Genre`)로 제한합니다. 이전에 `single` 선택을 사용했을 때는 선택이 개별 데이터 포인트에 매핑되었습니다. 선택을 특정 필드로 제한함으로써, `Major_Genre` 필드 값이 선택된 단일 값과 일치하는 *모든* 데이터 포인트를 선택할 수 있습니다.
- `init=...`을 사용하여 선택을 시작 값으로 초기화합니다.
- 선택을 인터페이스 위젯에 바인딩(`bind`)합니다. 이 경우 `binding_select`를 통한 드롭다운 메뉴입니다.
- 이전과 마찬가지로 조건부 인코딩을 사용하여 불투명도(opacity) 채널을 제어합니다.

### 여러 입력에 선택 바인딩하기 (Binding Selections to Multiple Inputs)

하나의 선택 인스턴스를 *여러 개의* 동적 쿼리 위젯에 바인딩할 수 있습니다. 위의 예제를 수정하여 메뉴 대신 라디오 버튼을 사용하고, `Major_Genre`와 `MPAA_Rating` *모두*에 대한 필터를 제공해 보겠습니다. 이제 우리의 `single` 선택은 장르와 MPAA 등급 값의 단일 *쌍(pair)*에 대해 정의됩니다.

*장르와 등급의 놀라운 결합을 찾아보세요. G 또는 PG 등급의 공포 영화가 있나요?*

In [10]:
# [Major_Genre, MPAA_Rating] 쌍에 대한 단일 값 선택
# 초기 선택된 값으로 특정 하드코딩된 값 사용
selection = alt.selection_single(
    name='Select',
    fields=['Major_Genre', 'MPAA_Rating'],
    init={'Major_Genre': 'Drama', 'MPAA_Rating': 'R'},
    bind={'Major_Genre': alt.binding_select(options=genres), 'MPAA_Rating': alt.binding_radio(options=mpaa)}
)
  
# 산점도, 선택에 따라 불투명도 수정
alt.Chart(movies).mark_circle().add_selection(
    selection
).encode(
    x='Rotten_Tomatoes_Rating:Q',
    y='IMDB_Rating:Q',
    tooltip='Title:N',
    opacity=alt.condition(selection, alt.value(0.75), alt.value(0.05))
)

alt.Chart(...)

*재미있는 사실: 영화 [죠스(Jaws)](https://www.imdb.com/title/tt0073195/)와 [죠스 2(Jaws 2)](https://www.imdb.com/title/tt0077766/)가 개봉했을 때는 PG-13 등급이 존재하지 않았습니다. 최초로 PG-13 등급을 받은 영화는 1984년의 [레드 던(Red Dawn)](https://www.imdb.com/title/tt0087985/)입니다.*

### 시각화를 동적 쿼리로 사용하기 (Using Visualizations as Dynamic Queries)

표준 인터페이스 위젯은 *가능한* 쿼리 매개변수 값을 보여주지만, 해당 값들의 *분포*를 시각화하지는 않습니다. 또한 한 번에 단일 값만 선택하는 입력 위젯보다는 다중 값 또는 간격 선택과 같은 더 풍부한 상호작용을 사용하고 싶을 수도 있습니다.

이러한 문제를 해결하기 위해 데이터를 시각화하는 동시에 동적 쿼리를 지원하는 추가 차트를 작성할 수 있습니다. 연도별 영화 수의 히스토그램을 추가하고, 선택된 기간 동안의 영화를 동적으로 강조 표시하기 위해 간격 선택(interval selection)을 사용해 보겠습니다.

*연도 히스토그램과 상호작용하여 다양한 시대의 영화를 탐색해 보세요. 연도에 따른 [샘플링 편향(sampling bias)](https://ko.wikipedia.org/wiki/%ED%91%9C%EB%B3%B8_%ED%8E%B8%ED%96%A5)의 증거가 보이나요? (개봉 연도와 비평가 평점은 어떤 관계가 있나요?)*

*연도가 1930년부터 2040년까지 걸쳐 있습니다! 미래의 영화가 사전 제작 중인 것일까요, 아니면 "1세기 차이" 오류가 있는 것일까요? 또한 거주하는 시간대에 따라 1969년 또는 1970년에 작은 돌출(bump)이 보일 수 있습니다. 왜 그럴까요? (노트북 끝부분의 설명을 참조하세요!)*

In [11]:
brush = alt.selection_interval(
    encodings=['x'] # 선택을 x축(연도) 값으로 제한
)

# 동적 쿼리 히스토그램
years = alt.Chart(movies).mark_bar().add_selection(
    brush
).encode(
    alt.X('year(Release_Date):T', title='개봉 연도별 영화'),
    alt.Y('count():Q', title=None)
).properties(
    width=650,
    height=50
)

# 산점도, 선택에 따라 불투명도 수정
ratings = alt.Chart(movies).mark_circle().encode(
    x='Rotten_Tomatoes_Rating:Q',
    y='IMDB_Rating:Q',
    tooltip='Title:N',
    opacity=alt.condition(brush, alt.value(0.75), alt.value(0.05))
).properties(
    width=650,
    height=400
)

alt.vconcat(years, ratings).properties(spacing=5)

alt.VConcatChart(...)

위의 예제는 차트 간의 *연결된 선택(linked selection)*을 사용하여 동적 쿼리를 제공합니다:

- 간격 선택(`brush`)을 생성하고, `encodings=['x']`를 설정하여 선택을 x축으로만 제한함으로써 1차원 선택 간격을 만듭니다.
- `.add_selection(brush)`를 통해 연도별 영화 히스토그램에 `brush`를 등록합니다.
- 산점도의 `opacity`(불투명도)를 조정하기 위해 조건부 인코딩에서 `brush`를 사용합니다.

한 차트에서 요소를 선택하고 하나 이상의 다른 차트에서 연결된 강조 표시를 보는 이 상호작용 기술을 [*브러싱 및 연결(brushing & linking)*](https://en.wikipedia.org/wiki/Brushing_and_linking)이라고 합니다.

## 이동 및 확대/축소 (Panning & Zooming)

영화 평점 산점도는 일부 구간이 다소 혼잡하여 밀도가 높은 지역의 점들을 조사하기 어렵습니다. *이동(panning)* 및 *확대/축소(zooming)* 상호작용 기술을 사용하면 밀집된 지역을 더 자세히 검사할 수 있습니다.

Altair 선택을 사용하여 이동 및 확대/축소를 표현하는 방법에 대해 생각해 봅시다. 차트의 "뷰포트(viewport)"를 정의하는 것은 무엇일까요? 바로 *축 스케일 도메인*입니다!

시각화된 데이터 값의 범위를 수정하기 위해 스케일 도메인을 변경할 수 있습니다. 이를 대화형으로 수행하기 위해 `bind='scales'` 코드를 사용하여 `interval` 선택을 스케일 도메인에 바인딩할 수 있습니다. 그 결과 드래그하고 줌할 수 있는 간격 브러시 대신, 전체 플로팅 영역을 드래그하고 줌할 수 있게 됩니다!

*아래 차트에서 클릭하고 드래그하여 뷰를 이동(이전)하거나, 스크롤하여 뷰를 확대/축소(스케일)해 보세요. 제공된 평점 값의 정밀도에 대해 무엇을 발견할 수 있나요?*

In [12]:
alt.Chart(movies).mark_circle().add_selection(
    alt.selection_interval(bind='scales')
).encode(
    x='Rotten_Tomatoes_Rating:Q',
    y=alt.Y('IMDB_Rating:Q', axis=alt.Axis(minExtent=30)), # 축 제목 배치를 안정화하기 위해 최소 범위 사용
    tooltip=['Title:N', 'Release_Date:N', 'IMDB_Rating:Q', 'Rotten_Tomatoes_Rating:Q']
).properties(
    width=600,
    height=400
)

alt.Chart(...)

*확대해 보면 평점 값의 정밀도가 제한적임을 알 수 있습니다! Rotten Tomatoes 평점은 정수이고, IMDB 평점은 소수점 첫째 자리에서 잘립니다. 결과적으로 확대하더라도 여러 영화가 동일한 평점 값을 공유하여 오버플로팅이 발생합니다.*

위의 코드를 읽어보면 `y` 인코딩 채널에서 `alt.Axis(minExtent=30)` 코드를 발견할 수 있습니다. `minExtent` 매개변수는 축 틱(tick)과 레이블을 위해 최소한의 공간을 예약하도록 보장합니다. 왜 이렇게 할까요? 이동하고 확대/축소할 때 축 레이블이 변경되어 축 제목 위치가 바뀔 수 있기 때문입니다. 최소 범위를 설정함으로써 플롯에서 주의를 분산시키는 움직임을 줄일 수 있습니다. *`minExtent` 값을 변경해 보세요(예: 0으로 설정). 그런 다음 축 레이블이 길어질 때 뷰가 어떻게 변하는지 확인하기 위해 축소해 보세요.*

Altair에는 플롯에 이동 및 확대/축소를 추가하기 위한 단축키도 포함되어 있습니다. 선택 항목을 직접 만드는 대신 `.interactive()`를 호출하여 Altair가 차트의 스케일에 바인딩된 간격 선택 항목을 자동으로 생성하도록 할 수 있습니다:

In [13]:
alt.Chart(movies).mark_circle().encode(
    x='Rotten_Tomatoes_Rating:Q',
    y=alt.Y('IMDB_Rating:Q', axis=alt.Axis(minExtent=30)), # 축 제목 배치를 안정화하기 위해 최소 범위 사용
    tooltip=['Title:N', 'Release_Date:N', 'IMDB_Rating:Q', 'Rotten_Tomatoes_Rating:Q']
).properties(
    width=600,
    height=400
).interactive()

alt.Chart(...)

기본적으로 선택 항목에 대한 스케일 바인딩은 `x` 및 `y` 인코딩 채널을 모두 포함합니다. 이동 및 확대/축소를 단일 차원으로만 제한하려면 어떻게 해야 할까요? `encodings=['x']`를 호출하여 선택 항목을 `x` 채널로만 제한할 수 있습니다:

In [14]:
alt.Chart(movies).mark_circle().add_selection(
    alt.selection_interval(bind='scales', encodings=['x'])
).encode(
    x='Rotten_Tomatoes_Rating:Q',
    y=alt.Y('IMDB_Rating:Q', axis=alt.Axis(minExtent=30)), # 축 제목 배치를 안정화하기 위해 최소 범위 사용
    tooltip=['Title:N', 'Release_Date:N', 'IMDB_Rating:Q', 'Rotten_Tomatoes_Rating:Q']
).properties(
    width=600,
    height=400
)

alt.Chart(...)

*단일 축을 따라 확대/축소할 때 시각화된 데이터의 모양이 변할 수 있으며, 이는 데이터의 관계에 대한 우리의 인식에 영향을 줄 수 있습니다. [적절한 종횡비(aspect ratio)를 선택하는 것](http://vis.stanford.edu/papers/arclength-banking)은 중요한 시각화 디자인 고려 사항입니다!*

## 탐색: 요약 + 세부 정보 (Navigation: Overview + Detail)

이동 및 확대/축소를 할 때 우리는 차트의 "뷰포트"를 직접 조정합니다. 이와 관련된 탐색 전략인 *요약 + 세부 정보(overview + detail)*는 모든 데이터를 보여주는 요약 디스플레이를 사용하는 대신, 별도의 초점 디스플레이를 이동하고 확대/축소하는 선택 항목을 지원합니다.

아래에는 S&P 500 주가 지수의 10년 동안의 가격 변동을 보여주는 두 개의 영역 차트가 있습니다. 처음에는 두 차트 모두 동일한 데이터 범위를 보여줍니다. *하단의 요약 차트에서 클릭하고 드래그하여 초점 디스플레이를 업데이트하고 특정 기간을 조사해 보세요.*

In [15]:
brush = alt.selection_interval(encodings=['x']);

base = alt.Chart().mark_area().encode(
    alt.X('date:T', title=None),
    alt.Y('price:Q')
).properties(
    width=700
)
  
alt.vconcat(
    base.encode(alt.X('date:T', title=None, scale=alt.Scale(domain=brush))),
    base.add_selection(brush).properties(height=60),
    data=sp500
)

alt.VConcatChart(...)

이전의 이동 및 확대/축소 사례와 달리, 여기서는 선택 항목을 단일 대화형 차트의 스케일에 직접 바인딩하고 싶지 않습니다. 대신 선택 항목을 *다른* 차트의 스케일 도메인에 바인딩하고 싶습니다. 이를 위해 초점 차트의 `x` 인코딩 채널을 업데이트하고 스케일 `domain` 속성이 `brush` 선택 항목을 참조하도록 설정합니다. 정의된 간격이 없으면(선택 항목이 비어 있으면) Altair는 브러시를 무시하고 기본 데이터를 사용하여 도메인을 결정합니다. 브러시 간격이 생성되면 Altair는 대신 이를 초점 차트의 스케일 `domain`으로 사용합니다.

## 세부 정보 표시 (Details on Demand)

시각화 내에서 관심 지점을 발견하면 종종 그에 대해 더 자세히 알고 싶어집니다. *세부 정보 표시(Details-on-demand)*는 선택된 값에 대해 더 많은 정보를 대화형으로 쿼리하는 것을 말합니다. *툴팁(Tooltips)*은 세부 정보를 제공하는 유용한 수단 중 하나입니다. 그러나 툴팁은 일반적으로 한 번에 하나의 데이터 포인트에 대한 정보만 보여줍니다. 어떻게 더 많은 정보를 보여줄 수 있을까요?

영화 평점 산점도에는 Rotten Tomatoes와 IMDB 평점이 일치하지 않는 잠재적으로 흥미로운 이상치가 많이 포함되어 있습니다. 점을 대화형으로 선택하고 해당 레이블을 보여주는 플롯을 만들어 보겠습니다. 호버(hover) 또는 클릭(click) 상호작용 중 하나에서 필터 쿼리를 트리거하기 위해 [Altair 구성 연산자](https://altair-viz.github.io/user_guide/interactions.html#composing-multiple-selections) `|` ("or")를 사용할 것입니다.

*아래 산점도의 점 위에 마우스를 올려 강조 표시와 제목 레이블을 확인하세요. 점을 Shift-클릭하면 주석이 유지되어 여러 레이블을 동시에 볼 수 있습니다. 어떤 영화가 Rotten Tomatoes 비평가들에게는 사랑받지만 IMDB의 일반 관객에게는 그렇지 않나요 (또는 그 반대)? 이름이 같은 두 개의 서로 다른 영화가 실수로 결합된 것으로 보이는 오류를 찾을 수 있는지 확인해 보세요!*

In [16]:
hover = alt.selection_single(
    on='mouseover',  # 마우스 오버 시 선택
    nearest=True,    # 마우스 커서와 가장 가까운 점 선택
    empty='none'     # 빈 선택 항목은 아무것도 일치하지 않아야 함
)

click = alt.selection_multi(
    empty='none' # 빈 선택 항목은 어떤 점과도 일치하지 않음
)

# 모든 마크가 공유하는 산점도 인코딩
plot = alt.Chart().mark_circle().encode(
    x='Rotten_Tomatoes_Rating:Q',
    y='IMDB_Rating:Q'
)
  
# 새로운 레이어를 위한 공유 베이스
base = plot.transform_filter(
    hover | click # 두 선택 항목 중 하나에 있는 점으로 필터링
)

# 레이어 산점도 점, 후광 주석 및 제목 레이블
alt.layer(
    plot.add_selection(hover).add_selection(click),
    base.mark_point(size=100, stroke='firebrick', strokeWidth=1),
    base.mark_text(dx=4, dy=-8, align='right', stroke='white', strokeWidth=2).encode(text='Title:N'),
    base.mark_text(dx=4, dy=-8, align='right').encode(text='Title:N'),
    data=movies
).properties(
    width=600,
    height=450
)

alt.LayerChart(...)

위의 예제는 산점도에 세 개의 새로운 레이어를 추가합니다: 원형 주석, 읽기 쉬운 배경을 제공하기 위한 흰색 텍스트, 영화 제목을 보여주는 검은색 텍스트입니다. 또한 이 예제는 두 개의 선택 항목을 병행하여 사용합니다:

1. 마우스가 움직임에 따라 가장 가까운 데이터 포인트를 자동으로 선택하기 위해 `nearest=True`를 포함하는 단일 선택(`hover`).
2. Shift-클릭을 통해 지속적인 선택 항목을 만들기 위한 다중 선택(`click`).

두 선택 항목 모두 선택 항목이 비어 있을 때 아무 점도 포함되지 않아야 함을 나타내기 위해 `empty='none'` 설정을 포함합니다. 그런 다음 이러한 선택 항목은 `hover`와 `click`의 논리적 *또는(or)*인 단일 필터 서술어로 결합되어, *두 선택 항목 중 하나*에 속하는 점을 포함합니다. 이 서술어를 사용하여 새로운 레이어를 필터링함으로써 선택된 점에 대해서만 주석과 레이블을 표시합니다.

선택 항목과 레이어를 사용하여 세부 정보 표시를 위한 다양한 디자인을 구현할 수 있습니다! 예를 들어, 다음은 로그 스케일이 적용된 기술주 주가의 시계열 데이터로, 마우스 커서와 가장 가까운 날짜에 가이드라인과 레이블이 표시되어 있습니다:

In [17]:
# 세부 정보 표시를 제공할 지점 선택
label = alt.selection_single(
    encodings=['x'], # 선택을 x축 값으로 제한
    on='mouseover',  # 마우스 오버 이벤트 시 선택
    nearest=True,    # 커서와 가장 가까운 데이터 포인트 선택
    empty='none'     # 빈 선택 항목은 데이터 포인트를 포함하지 않음
)

# 주가의 기본 선 차트 정의
base = alt.Chart().mark_line().encode(
    alt.X('date:T'),
    alt.Y('price:Q', scale=alt.Scale(type='log')),
    alt.Color('symbol:N')
)

alt.layer(
    base, # 기본 선 차트
    
    # 가이드라인 역할을 할 rule 마크 추가
    alt.Chart().mark_rule(color='#aaa').encode(
        x='date:T'
    ).transform_filter(label),
    
    # 선택된 시점에 대해 원 마크 추가, 선택되지 않은 점 숨기기
    base.mark_circle().encode(
        opacity=alt.condition(label, alt.value(1), alt.value(0))
    ).add_selection(label),

    # 레이블에 읽기 쉬운 배경을 제공하기 위해 흰색 테두리 텍스트 추가
    base.mark_text(align='left', dx=5, dy=-5, stroke='white', strokeWidth=2).encode(
        text='price:Q'
    ).transform_filter(label),

    # 주가에 대한 텍스트 레이블 추가
    base.mark_text(align='left', dx=5, dy=-5).encode(
        text='price:Q'
    ).transform_filter(label),
    
    data=stocks
).properties(
    width=700,
    height=400
)

alt.LayerChart(...)

*지금까지 배운 내용을 실천에 옮겨보세요: 위에서 보았던 영화 산점도(연도에 따른 동적 쿼리가 있는 것)를 수정하여, 연도 `interval` 선택 내에 포함된 데이터의 평균 IMDB(또는 Rotten Tomatoes) 평점을 보여주는 `rule` 마크를 포함할 수 있습니까?*

## 브러싱 및 연결 재방문 (Brushing & Linking, Revisited)

이 노트북의 앞부분에서 우리는 *브러싱 및 연결(brushing & linking)*의 예를 보았습니다: 동적 쿼리 히스토그램을 사용하여 영화 평점 산점도의 점들을 강조 표시하는 것이었습니다. 여기서는 연결된 선택 항목과 관련된 추가 예시를 살펴보겠습니다.

`cars` 데이터 세트로 돌아가서, `repeat` 연산자를 사용하여 연비, 가속력, 마력 간의 연관성을 보여주는 [산점도 행렬(SPLOM)](https://en.wikipedia.org/wiki/Scatter_plot#Scatterplot_matrices)을 구성할 수 있습니다. `interval` 선택을 정의하고 이를 반복되는 산점도 사양 *내*에 포함하여 모든 플롯 간에 연결된 선택을 활성화할 수 있습니다.

*아래 플롯 중 하나에서 클릭하고 드래그하여 브러싱 및 연결을 수행해 보세요!*

In [18]:
brush = alt.selection_interval(
    resolve='global' # 모든 선택 항목을 단일 글로벌 인스턴스로 해결
)

alt.Chart(cars).mark_circle().add_selection(
    brush
).encode(
    alt.X(alt.repeat('column'), type='quantitative'),
    alt.Y(alt.repeat('row'), type='quantitative'),
    color=alt.condition(brush, 'Cylinders:O', alt.value('grey')),
    opacity=alt.condition(brush, alt.value(0.8), alt.value(0.1))
).properties(
    width=140,
    height=140
).repeat(
    column=['Acceleration', 'Horsepower', 'Miles_per_Gallon'],
    row=['Miles_per_Gallon', 'Horsepower', 'Acceleration']
)

alt.RepeatChart(...)

위에서 `interval` 선택에 `resolve='global'`을 사용한 점에 주목하세요. 기본 설정인 `'global'`은 모든 플롯에서 한 번에 하나의 브러시만 활성화될 수 있음을 나타냅니다. 그러나 어떤 경우에는 여러 플롯에서 브러시를 정의하고 결과를 결합하고 싶을 수도 있습니다. `resolve='union'`을 사용하면 선택 항목은 모든 브러시의 *합집합(union)*이 됩니다: 점이 브러시 중 하나라도 속하면 선택됩니다. 또는 `resolve='intersect'`를 사용하면 선택 항목은 모든 브러시의 *교집합(intersection)*으로 구성됩니다: 모든 브러시 내에 속하는 점들만 선택됩니다.

*`resolve` 매개변수를 `'union'` 및 `'intersect'`로 설정해 보고 결과로 나오는 선택 논리가 어떻게 변하는지 확인해 보세요.*

### 교차 필터링 (Cross-Filtering)

지금까지 살펴본 브러싱 및 연결 예제는 모두 조건부 인코딩을 사용하여 선택에 반응하여 불투명도 값을 변경하는 등의 작업을 수행했습니다. 또 다른 옵션은 한 뷰에서 정의된 선택 항목을 사용하여 다른 뷰의 내용을 *필터링*하는 것입니다.

`flights` 데이터 세트에 대한 히스토그램 컬렉션을 구성해 보겠습니다: 도착 지연(`delay`, 분 단위), 비행 거리(`distance`, 마일 단위), 출발 시간(`time`, 하루 중 시간). `repeat` 연산자를 사용하여 히스토그램을 생성하고, 교집합(`intersect`)을 통해 해결된 브러시와 함께 x축에 대한 `interval` 선택을 추가하겠습니다.

특히 각 히스토그램은 회색 배경 레이어와 파란색 전경 레이어의 두 레이어로 구성되며, 전경 레이어는 브러시 선택의 교집합에 의해 필터링됩니다. 그 결과 세 차트 간에 *교차 필터링* 상호작용이 발생합니다!

*아래 차트에서 브러시 간격을 드래그해 보세요. 도착 지연 시간이 길거나 짧은 항공편을 선택함에 따라 비행 거리와 시간 분포가 어떻게 반응하나요?*

In [19]:
brush = alt.selection_interval(
    encodings=['x'],
    resolve='intersect'
);

hist = alt.Chart().mark_bar().encode(
    alt.X(alt.repeat('row'), type='quantitative',
        bin=alt.Bin(maxbins=100, minstep=1), # 최대 100개 구간
        axis=alt.Axis(format='d', titleAnchor='start') # 정수 형식, 왼쪽 정렬된 제목
    ),
    alt.Y('count():Q', title=None) # y축 제목 없음
)
  
alt.layer(
    hist.add_selection(brush).encode(color=alt.value('lightgrey')),
    hist.transform_filter(brush)
).properties(
    width=900,
    height=100
).repeat(
    row=['delay', 'distance', 'time'],
    data=flights
).transform_calculate(
    delay='datum.delay < 180 ? datum.delay : 180', # 3시간 이상의 지연 제한
    time='hours(datum.date) + minutes(datum.date) / 60' # 소수점 시간
).configure_view(
    stroke='transparent' # 외곽선 없음
)

alt.RepeatChart(...)

*교차 필터링을 통해 지연된 항공편이 늦은 시간에 출발할 가능성이 더 높다는 것을 관찰할 수 있습니다. 이러한 현상은 자주 비행기를 타는 사람들에게 익숙한데, 지연이 하루 종일 전파되어 해당 항공기의 이후 여행에 영향을 미칠 수 있기 때문입니다. 정시 도착 확률을 높이려면 이른 시간의 항공편을 예약하세요!*

멀티 뷰와 대화형 선택의 결합은 가치 있는 형태의 다차원적 추론을 가능하게 하여, 기본적인 히스토그램조차 데이터 세트에 질문을 던지기 위한 강력한 입력 장치로 변화시킵니다!

## 요약 (Summary)

Altair에서 지원하는 상호작용 옵션에 대한 자세한 내용은 [Altair 대화형 선택 문서](https://altair-viz.github.io/user_guide/interactions.html)를 참조하세요. 여러 상호작용 기술을 구성하거나 모바일 장치에서의 터치 기반 입력을 지원하기 위한 이벤트 핸들러 사용자 정의에 대한 세부 사항은 [Vega-Lite 선택 문서](https://vega.github.io/vega-lite/docs/selection.html)를 참조하세요.

더 자세히 알고 싶으신가요?
- *선택(selection)* 추상화는 Satyanarayan, Moritz, Wongsuphasawat, & Heer의 논문 [Vega-Lite: A Grammar of Interactive Graphics](http://idl.cs.washington.edu/papers/vega-lite/)에서 소개되었습니다.
- PRIM-9 시스템(최대 9차원에서의 투영, 회전, 격리 및 마스킹을 위한 시스템)은 1970년대 초 Fisherkeller, Tukey, & Friedman에 의해 구축된 최초의 대화형 시각화 도구 중 하나입니다. [복고풍 데모 비디오가 남아 있습니다!](https://www.youtube.com/watch?v=B7XoW2qiFUA)
- 브러싱 및 연결의 개념은 Becker, Cleveland, & Wilks의 1987년 기사 [Dynamic Graphics for Data Analysis](https://scholar.google.com/scholar?cluster=14817303117298653693)에서 구체화되었습니다.
- 시각화를 위한 상호작용 기술의 포괄적인 요약은 Heer & Shneiderman의 [Interactive Dynamics for Visual Analysis](https://queue.acm.org/detail.cfm?id=2146416)를 참조하세요.
- 마지막으로 상호작용을 효과적으로 만드는 요소에 대한 논문으로 Hutchins, Hollan, & Norman의 고전적인 [Direct Manipulation Interfaces](https://scholar.google.com/scholar?cluster=15702972136892195211)를 읽어보세요.

#### 부록: 시간의 표현에 대하여 (Appendix: On The Representation of Time)

앞서 우리는 1969년 또는 1970년에 영화 수가 작게 돌출된 것을 관찰했습니다. 그 돌출은 어디에서 온 것일까요? 그리고 왜 1969년 *또는* 1970년일까요? 그 답은 누락된 데이터와 컴퓨터가 시간을 표현하는 방식의 결합에서 기인합니다.

내부적으로 날짜와 시간은 [UNIX 시간(epoch)](https://ko.wikipedia.org/wiki/유닉스_시간)을 기준으로 표현되는데, 여기서 시간 "0"은 [본초 자오선](https://ko.wikipedia.org/wiki/%EB%B3%B8%EC%B4%88_%EC%9E%90%EC%98%A4%EC%84%A0)을 따라 흐르는 [UTC 시간](https://ko.wikipedia.org/wiki/%ED%98%91%EC%A0%95_%EC%84%B8%EA%B3%84%EC%8B%9C)으로 1970년 1월 1일 자정에 해당합니다. 알고 보니 개봉일이 누락된(`null`) 영화가 몇 편 있습니다. 이러한 `null` 값은 시간 `0`으로 해석되어 UTC 시간으로 1970년 1월 1일에 매핑됩니다. 만약 당신이 아메리카 대륙에 살고 있다면(따라서 "더 빠른" 시간대에 있다면), 이 정확한 시점은 현지 시간대로 1969년 12월 31일의 이른 시간에 해당합니다. 반면에 본초 자오선 근처나 그 동쪽에 살고 있다면 현지 시간대의 날짜는 1970년 1월 1일이 됩니다.

교훈은 무엇일까요? 항상 데이터를 의심하고, 데이터가 표현되는 방식(날짜 시간, 부동 소수점 숫자, 위도와 경도 등)이 때때로 분석에 영향을 미치는 인공적인 결과(artifact)를 초래할 수 있음을 명심하세요!